In [1]:
import numpy as np
from parameters import get_parameters, get_slider_params, calculate_derived_parameters
from model_run import run_model_dash
from global_func import reset_flags, reset_E, reset_HSS, reset_S
import math
import matplotlib.pyplot as plt
import pandas as pd
from skopt import gp_minimize
from skopt.space import Real
from skopt.utils import use_named_args

In [2]:
total_runs = 100
seeds = np.random.default_rng(2025).integers(low=0, high=1e6, size=total_runs)
MODEL = {
    "int_period": 36,
    "n_months": 36,
}
slider_params = get_slider_params()
results = []
for run in range(total_runs):
    base_seed = seeds[run]
    rng_param = np.random.default_rng(base_seed)
    b_param = get_parameters(rng = rng_param)
    b_param = calculate_derived_parameters(b_param)
    b_flags = reset_flags()
    b_HSS = reset_HSS(slider_params)
    b_S = reset_S(slider_params)
    b_E = reset_E()
    b_param.update({"E": b_E, "S": b_S, "HSS": b_HSS
    })
    n_months = MODEL["n_months"]
    int_period = MODEL["int_period"]
    _, outcomes = run_model_dash(b_param, b_flags, n_months, int_period, base_seed=base_seed)

    # Calculate outcomes
    outcomes['i_facility'] = np.where(outcomes['i_loc_new_v2'] > 0, 1, 0)
    outcomes['i_loc_grouped'] = np.where(outcomes['i_loc_new_v2'] == 0, 0,
                                         np.where(outcomes['i_loc_new_v2'] == 1, 1, 2))

    results.append({
        "p_elcs_fac": outcomes.groupby('i_facility')['i_elec_CS'].mean().get(1, 0),
        'p_elcs_highrisk_l45': outcomes[(outcomes['i_loc_grouped'] == 2) & (outcomes['i_risk'] == 1)]['i_elec_CS'].mean(),
        "p_elcs_preterm_l45": outcomes[(outcomes['i_loc_grouped'] == 2) & (outcomes['i_preterm'] == 1)]['i_elec_CS'].mean(),
    })

    # Convert to DataFrame and compute mean
df_results = pd.DataFrame(results).mean().to_dict()
print(df_results)

{'p_elcs_fac': 0.06593321933403735, 'p_elcs_highrisk_l45': 0.09673505671851888, 'p_elcs_preterm_l45': 0.2670414190121392}


In [3]:
# Define the parameter space for Bayesian Optimization
param_space = [
    Real(0.0, 0.2, name='p_elec_CS|highrisk'),
    Real(0.5, 0.8, name='p_elec_CS|preterm'),
]

# Calibration target values
target_values = {
    "p_elcs_fac": 0.066,
    "p_elcs_preterm_l45": 0.265,
}

# Generate unique seeds for reproducibility
total_runs = 100
seeds = np.random.default_rng(2025).integers(low=0, high=1e6, size=total_runs)

# Objective function for Bayesian Optimization
@use_named_args(param_space)
def objective(**params):

    from parameters import get_parameters, get_slider_params
    from model_run import run_model_dash
    from global_func import reset_flags, reset_E, reset_HSS, reset_S

    MODEL = {
        "int_period": 36,
        "n_months": 36,
    }

    slider_params = get_slider_params()
    results = []

    for run in range(total_runs):
        base_seed = seeds[run]
        rng_param = np.random.default_rng(base_seed)
        b_param = get_parameters(rng=rng_param)
        b_flags = reset_flags()
        b_HSS = reset_HSS(slider_params)
        b_S = reset_S(slider_params)
        b_E = reset_E()
        b_param.update({"E": b_E, "S": b_S, "HSS": b_HSS})

        # Replace parameters with those to calibrate
        b_param["p_elec_CS|highrisk"] = params["p_elec_CS|highrisk"]
        b_param["p_elec_CS|preterm"] = params["p_elec_CS|preterm"]

        #update parameters
        b_param = calculate_derived_parameters(b_param)

        #run the model
        n_months = MODEL["n_months"]
        int_period = MODEL["int_period"]
        _, outcomes = run_model_dash(b_param, b_flags, n_months, int_period, base_seed=base_seed)

        # Calculate outcomes
        outcomes['i_facility'] = np.where(outcomes['i_loc_new_v2'] > 0, 1, 0)
        outcomes['i_loc_grouped'] = np.where(outcomes['i_loc_new_v2'] == 0, 0,
                                             np.where(outcomes['i_loc_new_v2'] == 1, 1, 2))

        results.append({
            "p_elcs_fac": outcomes.groupby('i_facility')['i_elec_CS'].mean().get(1, 0),
            "p_elcs_preterm_l45": outcomes[(outcomes['i_loc_grouped'] == 2) & (outcomes['i_preterm'] == 1)]['i_elec_CS'].mean(),
        })

    # Convert to DataFrame and compute mean
    df_results = pd.DataFrame(results).mean().to_dict()

    # Add this line here for early stopping access
    objective.last_df_results = df_results

    relative_squared_errors = [ ((df_results[k] - target_values[k]) / target_values[k])**2 for k in target_values ]
    rmse = math.sqrt(sum(relative_squared_errors) / len(relative_squared_errors))

    # Print current evaluation
    print("\n--- Calibration Iteration ---")
    print("Parameters:")
    for k, v in params.items():
        print(f"  {k:22s}: {v:.4f}")

    print("\nTarget vs Simulated Outcomes:")
    for k in target_values:
        sim = df_results.get(k, 0)
        target = target_values[k]
        error = sim - target
        rel_error = error / target
        print(f"  {k:22s} | Simulated: {sim:.4f}, Target: {target:.4f}, Relative Error: {rel_error:+.2%}")


    print(f"\nRMSE Loss: {rmse:.6f}")
    print("-----------------------------\n")

    return rmse

In [4]:
# Track RMSE history
rmse_history = []

class EarlyStopper:
    def __init__(self, threshold_rmse=0.05, relative_threshold=0.05):
        self.threshold_rmse = threshold_rmse
        self.relative_threshold = relative_threshold
        self.best_loss = float('inf')
        self.call_count = 0
        self.best_params = None
        self.best_df_results = None

    def meets_relative_criteria(self, sim_results):
        for k in target_values:
            target = target_values[k]
            sim = sim_results.get(k, 0)
            rel_error = abs(sim - target) / target
            if rel_error > self.relative_threshold:
                return False
        return True

    def __call__(self, *args, **kwargs):
        loss = objective(*args, **kwargs)
        rmse_history.append(loss)

        self.call_count += 1
        if loss < self.best_loss:
            self.best_loss = loss
            self.best_params = args[0] if args else None

        sim_results = getattr(objective, "last_df_results", {})
        if self.best_loss < self.threshold_rmse and self.meets_relative_criteria(sim_results):
            print(f"\n⏹️ Early stopping triggered. RMSE: {self.best_loss:.6f}")
            raise StopIteration

        return loss

def save_results(best_params, best_loss, rmse_history, param_space):
    param_names = [dim.name for dim in param_space]
    best_param_dict = dict(zip(param_names, best_params))
    best_param_dict["Best RMSE"] = best_loss
    df_best = pd.DataFrame([best_param_dict])
    df_best.to_csv("best_params_ELCS_cal.csv", index=False)

    pd.DataFrame({"Iteration": range(1, len(rmse_history)+1), "RMSE": rmse_history}).to_csv(
        "rmse_history_ELCS_cal.csv", index=False)

    plt.figure(figsize=(10, 6))
    plt.plot(range(1, len(rmse_history)+1), rmse_history, marker='o')
    plt.xlabel("Iteration")
    plt.ylabel("RMSE Loss")
    plt.title("RMSE Loss Over Iterations")
    plt.grid(True)
    plt.tight_layout()
    plt.savefig("rmse_plot_ELCS_cal.png")
    plt.close()

In [5]:
def optimize_model():
    early_stopper = EarlyStopper(threshold_rmse=0.05, relative_threshold=0.05)
    try:
        result = gp_minimize(
            func=early_stopper,
            dimensions=param_space,
            n_calls=100,
            random_state=42,
            n_random_starts=10,
            verbose=True
        )
        print("\n✅ Optimization completed.")
        print("Best Parameters:", result.x)
        print("Best Loss:", result.fun)
        save_results(result.x, result.fun, rmse_history, param_space)
        return result

    except StopIteration:
        print("\n✅ Optimization stopped early.")
        print(f"Best RMSE achieved: {early_stopper.best_loss:.6f}")
        if early_stopper.best_params is not None:
            print("Best Parameters:")
            for name, val in zip([d.name for d in param_space], early_stopper.best_params):
                print(f"  {name:22s}: {val:.4f}")
            save_results(early_stopper.best_params, early_stopper.best_loss, rmse_history, param_space)
        return None
#
# Run the optimization
if __name__ == "__main__":
    optimize_model()

Iteration No: 1 started. Evaluating function at random point.

--- Calibration Iteration ---
Parameters:
  p_elec_CS|highrisk    : 0.1593
  p_elec_CS|preterm     : 0.5550

Target vs Simulated Outcomes:
  p_elcs_fac             | Simulated: 0.0728, Target: 0.0660, Relative Error: +10.29%
  p_elcs_preterm_l45     | Simulated: 0.1972, Target: 0.2650, Relative Error: -25.59%

RMSE Loss: 0.195014
-----------------------------

Iteration No: 1 ended. Evaluation done at random point.
Time taken: 88.5865
Function value obtained: 0.1950
Current minimum: 0.1950
Iteration No: 2 started. Evaluating function at random point.

--- Calibration Iteration ---
Parameters:
  p_elec_CS|highrisk    : 0.1559
  p_elec_CS|preterm     : 0.6791

Target vs Simulated Outcomes:
  p_elcs_fac             | Simulated: 0.0799, Target: 0.0660, Relative Error: +21.07%
  p_elcs_preterm_l45     | Simulated: 0.2365, Target: 0.2650, Relative Error: -10.74%

RMSE Loss: 0.167235
-----------------------------

Iteration No: 2 

In [ ]:
# --- Calibration Iteration ---
# Parameters:
#   p_elec_CS|highrisk    : 0.0812
#   p_elec_CS|preterm     : 0.8000
#
# Target vs Simulated Outcomes:
#   p_elcs_fac             | Simulated: 0.0707, Target: 0.0660, Relative Error: +7.05%
#   p_elcs_preterm_l45     | Simulated: 0.2692, Target: 0.2650, Relative Error: +1.58%
#
# RMSE Loss: 0.051105
# -----------------------------
#
# Iteration No: 15 ended. Search finished for the next optimal point.
# Time taken: 87.3323
# Function value obtained: 0.0511
# Current minimum: 0.0511
# Iteration No: 16 started. Searching for the next optimal point.
#
# --- Calibration Iteration ---
# Parameters:
#   p_elec_CS|highrisk    : 0.0609
#   p_elec_CS|preterm     : 0.8000
#
# Target vs Simulated Outcomes:
#   p_elcs_fac             | Simulated: 0.0659, Target: 0.0660, Relative Error: -0.10%
#   p_elcs_preterm_l45     | Simulated: 0.2670, Target: 0.2650, Relative Error: +0.77%
#
# RMSE Loss: 0.005494
# -----------------------------
#
#
# ⏹️ Early stopping triggered. RMSE: 0.005494
#
# ✅ Optimization stopped early.
# Best RMSE achieved: 0.005494
# Best Parameters:
#   p_elec_CS|highrisk    : 0.0609
#   p_elec_CS|preterm     : 0.8000